# 01 · Setup: folders, central config, environment, smoke tests
**Purpose:** run **once** at project start (and again only if you intentionally change the config).
Creates the Drive folder tree, writes the central `config.json`, registers the provided Kazakh data (COLING-2025 Kazakh subset) with a checksum, and smoke-tests every external dependency (KerasHub presets, OpenRouter, Wikipedia API, GlotLID, HF scorer models).

**Runtime:** CPU is enough. **Inputs:** `data/data/coling2025_kazakh.jsonl`. **Outputs:** `00_admin/config/config.json`, `01_raw/external/…`, `00_admin/env/*.json`.

**Before running:** add Colab secrets (🔑 icon, left bar): `OPENROUTER_API_KEY` (required), `HF_TOKEN` (optional).

In [ ]:
# ==== Bootstrap (no common library yet: this notebook creates the config it needs) ====
NOTEBOOK = "01_setup_environment"
PROJECT_ROOT = "/content/drive/MyDrive/multilingual-ai-text-detection"   # <- edit if the folder lives elsewhere
import os, json, shutil, hashlib
from pathlib import Path
try:
    from google.colab import drive; drive.mount("/content/drive")
except ImportError:
    pass
ROOT = Path(PROJECT_ROOT); assert ROOT.exists(), f"Project folder not found: {ROOT}"

In [ ]:
# Pinned stack (verified on PyPI 2026-10-02; Colab default TF 2.21.0 / tensorflow-text 2.21.1)
!pip -q install "keras-hub==0.32.0" "keras==3.15.1" datasketch fasttext huggingface_hub

## Central configuration
Edit **here only**. Bump `config_version` whenever an experimental setting changes (it is part of every run ID, so old results can never silently mix with new ones).

In [ ]:
TURKIC = ["kk", "ky", "uz", "tr"]
ALL_LANGS = ["kk", "ky", "uz", "tr", "ru", "en"]

CONFIG = {
  "config_version": "1.2",
  "data_version": "v1",
  "created_utc": None,
  "paths": {
    "config": "00_admin/config", "logs": "00_admin/logs", "env": "00_admin/env", "registry": "00_admin/registry",
    "raw_external": "01_raw/external", "raw_wiki": "01_raw/wikipedia", "raw_gen": "01_raw/generations",
    "validated": "02_validated", "processed": "03_processed", "splits": "04_splits", "attacks": "05_attacks",
    "models": "06_models", "predictions": "07_predictions", "results": "08_results", "stats": "09_statistics",
    "figures": "10_figures_tables/figures", "tables": "10_figures_tables/tables", "manuscript": "11_manuscript",
    "notebooks": "notebooks"},
  # 2x2 factorial: family (Turkic vs Indo-European) x script (Cyrillic vs Latin)
  "languages": {
    "kk": {"name": "Kazakh",  "wiki": "kk", "family": "Turkic",     "script": "Cyrillic", "lid": "kaz_Cyrl", "nllb": "kaz_Cyrl"},
    "ky": {"name": "Kyrgyz",  "wiki": "ky", "family": "Turkic",     "script": "Cyrillic", "lid": "kir_Cyrl", "nllb": "kir_Cyrl"},
    "uz": {"name": "Uzbek",   "wiki": "uz", "family": "Turkic",     "script": "Latin",    "lid": "uzn_Latn", "nllb": "uzn_Latn"},
    "tr": {"name": "Turkish", "wiki": "tr", "family": "Turkic",     "script": "Latin",    "lid": "tur_Latn", "nllb": "tur_Latn"},
    "ru": {"name": "Russian", "wiki": "ru", "family": "Indo-European", "branch": "Slavic",   "script": "Cyrillic", "lid": "rus_Cyrl", "nllb": "rus_Cyrl"},
    "en": {"name": "English", "wiki": "en", "family": "Indo-European", "branch": "Germanic", "script": "Latin",    "lid": "eng_Latn", "nllb": "eng_Latn"}},
  "turkic_targets": TURKIC,
  "data": {
    "wiki_cutoff_utc": "2022-11-01T00:00:00Z",      # human texts = article revision as of this date (pre-ChatGPT)
    "n_candidates_per_lang": 4000, "n_topics_target": 1000, "gen_oversample": 1.25, "gen_margin": 1.15,
    "wiki_min_bytes": 2500, "template_df": 5, "template_max_share": 0.3, "max_topup_rounds": 4,
    "length_tol_words": 10, "length_tol_frac": 0.25,
    "min_words": 40, "max_words": 200, "strip_parentheticals": True,           # unit = one paragraph (matches COLING-2025 Kazakh format)
    "langid_min_prob": 0.5, "minhash_threshold": 0.8, "minhash_perm": 128,
    "split_pct": {"train": 70, "val": 10, "test": 20},
    "user_agent": "MATD-research/1.0 (SDU University; academic research)"},
  # 5 generator families via OpenRouter. Qwen is deliberately NOT a generator (zero-shot scorers are Qwen-based).
  "generators": {
    "gpt41mini":   {"model": "openai/gpt-4.1-mini",                 "family": "openai"},
    "haiku45":     {"model": "anthropic/claude-haiku-4.5",          "family": "anthropic"},
    "gem25lite":   {"model": "google/gemini-2.5-flash-lite",        "family": "google"},
    "llama33_70b": {"model": "meta-llama/llama-3.3-70b-instruct",   "family": "meta"},
    "dsv3":        {"model": "deepseek/deepseek-chat-v3-0324",      "family": "deepseek"}},
  "generation": {"temperature": 0.7, "top_p": 0.95, "max_tokens": 700, "n_workers": 8,
                 "budget_cap_usd": 150.0, "timeout_s": 120,
                 # prompt templates mirror COLING-2025 Task 1 Kazakh data (verified from the provided file)
                 "prompts": {
                   "p0": "Please, write one paragraph about the following topic in {lang_name}: {title}.",
                   "p1": "Please, write one paragraph in {lang_name} about the following topic: {title}. Avoid repetitive sentence structures and predictable phrasing. When applicable, include concrete facts such as specific numbers, dates, or historical references. Avoid overly flattering or exaggerated language, and instead focus on delivering clear, informative, and relevant content."}},
  "attacks": {
    "T1_paraphrase": {"model": "qwen/qwen3-235b-a22b-2507", "temperature": 0.7, "max_tokens": 900,
                      "prompt": "Paraphrase the following {lang_name} text in {lang_name}. Keep the meaning and roughly the same length, but change the wording and sentence structure. Output only the paraphrased text.\n\n{text}"},
    "T2_backtranslation": {"model": "facebook/nllb-200-distilled-600M", "pivot": {"default": "eng_Latn", "en": "rus_Cyrl"}},
    "T3_homoglyph": {"levels": [0.05, 0.15]},
    "T4_typo": {"levels": [0.05, 0.15]},
    "similarity_model": "sentence-transformers/LaBSE"},
  "seeds": [13, 42, 2026],
  "finetune": {
    "backend": "tensorflow",
    "models": {"mdeberta": {"class": "DebertaV3TextClassifier", "preset": "deberta_v3_base_multi"},
               "xlmr":     {"class": "XLMRobertaTextClassifier", "preset": "xlm_roberta_base_multi"}},
    "seq_len": 256, "batch_size": 32, "epochs": 3, "lr": 2e-5, "weight_decay": 0.01, "warmup_frac": 0.1,
    "early_stopping_patience": 1,
    "save_checkpoint_for": ["E2__all"]},          # keep weights only for the main multilingual models
  "ngram": {"analyzer": "char_wb", "ngram_range": [1, 5], "max_features": 300000, "C_grid": [0.1, 1.0, 10.0]},
  "zeroshot": {
    "pairs": {"generic": {"observer": "Qwen/Qwen3.5-4B-Base",        "performer": "Qwen/Qwen3.5-4B"},
              "kazakh":  {"observer": "issai/Qwen3.5-4B-Base-Kazakh", "performer": "issai/Qwen3.5-4B-Kazakh"}},
    "max_tokens": 256, "batch_size": 4, "dtype": "bfloat16"},
  "experiments": {
    "E1": [{"name": l, "train_langs": [l]} for l in ALL_LANGS],
    "E2": [{"name": "all", "train_langs": ALL_LANGS}] + [{"name": f"no-{l}", "train_langs": [x for x in ALL_LANGS if x != l]} for l in TURKIC],
    "E3": [{"name": "all", "train_langs": ALL_LANGS, "heldout_family": f} for f in ["openai", "anthropic", "google", "meta", "deepseek"]],
    "E6": [{"name": "all-aug", "train_langs": ALL_LANGS, "augment": ["T3_homoglyph", "T4_typo"], "optional": True}]},
  "stats": {"bootstrap_B": 2000, "alpha": 0.05, "fpr_points": [0.01, 0.05]},
  "external": {"coling_kk_source": "data/data/coling2025_kazakh.jsonl"},
  "tools": {"glotlid_repo": "cis-lmu/glotlid"},
}

In [ ]:
from datetime import datetime, timezone
cfg_dir = ROOT / CONFIG["paths"]["config"]; cfg_dir.mkdir(parents=True, exist_ok=True)
cfg_path = cfg_dir / "config.json"
FORCE_OVERWRITE = False      # set True only when deliberately changing the design (and bump config_version)

if cfg_path.exists() and not FORCE_OVERWRITE:
    old = json.load(open(cfg_path))
    print("config.json exists (version", old["config_version"], ") — kept. Set FORCE_OVERWRITE=True to replace.")
else:
    if cfg_path.exists():   # archive the previous version
        old = json.load(open(cfg_path))
        shutil.copy(cfg_path, cfg_dir / f"config_v{old['config_version']}_{datetime.now():%Y%m%d%H%M%S}.json")
    CONFIG["created_utc"] = datetime.now(timezone.utc).isoformat()
    json.dump(CONFIG, open(cfg_path, "w"), indent=1, ensure_ascii=False)
    print("wrote", cfg_path)

for rel in json.load(open(cfg_path))["paths"].values():
    (ROOT / rel).mkdir(parents=True, exist_ok=True)
print("folder tree ready")

## Load the common library (now that the config exists)

In [ ]:
os.environ["MATD_ROOT"] = PROJECT_ROOT
def load_common(path=f"{PROJECT_ROOT}/notebooks/00_common.ipynb"):
    nb = json.load(open(path, encoding="utf-8"))
    code = "\n\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
    exec(compile(code, path, "exec"), globals())
load_common()
log = get_logger(NOTEBOOK); env = env_snapshot(NOTEBOOK)
log.info(f"GPU: {env['gpu']}")

## Register the provided Kazakh data (COLING-2025 GenAI Detection Task 1, Kazakh test subset)
Copied (never moved) into `01_raw/external/` with a SHA-256 so every later step can verify it.

In [ ]:
src = Path(PROJECT_ROOT) / CFG["external"]["coling_kk_source"]
dst = P["raw_external"] / "coling2025_kazakh.jsonl"
if not dst.exists():
    shutil.copy(src, dst)
manifest = {"file": str(dst.relative_to(PROJECT_ROOT)), "sha256": sha256_file(dst), "source_copy_of": str(src.relative_to(PROJECT_ROOT)),
            "origin": "COLING 2025 GenAI Detection Task 1 (multilingual) — Kazakh test items; HF: Jinyan1/COLING_2025_MGT_multingual (Apache-2.0 repo)",
            "registered_utc": utcnow()}
write_json(P["raw_external"] / "coling2025_kazakh.manifest.json", manifest)
kk_ext = pd.DataFrame(read_jsonl(dst))
log.info(f"COLING-kk rows={len(kk_ext)} labels={kk_ext.label.value_counts().to_dict()} models={kk_ext.model.value_counts().to_dict()}")

## Smoke tests (each prints OK / a clear error). Nothing here is cached; re-run any time.

In [ ]:
# 1) KerasHub presets for the two fine-tuned detectors (downloads tokenizer config only)
os.environ["KERAS_BACKEND"] = CFG["finetune"]["backend"]
import keras, keras_hub, tensorflow as tf
log.info(f"keras {keras.__version__} | keras_hub {keras_hub.__version__} | tf {tf.__version__} | backend {keras.backend.backend()}")
for name, m in CFG["finetune"]["models"].items():
    cls = getattr(keras_hub.models, m["class"])
    assert m["preset"] in cls.presets, f"{m['preset']} not in {cls.__name__}.presets"
    pre = cls.preprocessor_cls.from_preset(m["preset"], sequence_length=16)
    out = pre(["Сәлеметсіз бе, әлем!"])
    log.info(f"OK KerasHub {name}: {m['preset']} -> token_ids shape {tuple(out['token_ids'].shape) if isinstance(out, dict) else 'ok'}")

In [ ]:
# 2) OpenRouter key + one tiny call per generator (≈ $0.001 total)
import requests
from google.colab import userdata
OR_KEY = userdata.get("OPENROUTER_API_KEY")
def or_chat(model, prompt, max_tokens=30, **kw):
    r = requests.post("https://openrouter.ai/api/v1/chat/completions",
                      headers={"Authorization": f"Bearer {OR_KEY}", "Content-Type": "application/json"},
                      json={"model": model, "messages": [{"role": "user", "content": prompt}],
                            "max_tokens": max_tokens, "usage": {"include": True}, **kw}, timeout=60)
    r.raise_for_status(); return r.json()
for gid, g in {**CFG["generators"], "paraphraser": {"model": CFG["attacks"]["T1_paraphrase"]["model"]}}.items():
    try:
        j = or_chat(g["model"], "Say 'сәлем' in one word.")
        log.info(f"OK OpenRouter {gid}: provider={j.get('provider')} usage={j.get('usage')}")
    except Exception as e:
        log.error(f"FAIL OpenRouter {gid}: {e}")

In [ ]:
# 3) Wikipedia API: revision-as-of-cutoff lookup for one Kazakh article
r = requests.get("https://kk.wikipedia.org/w/api.php", headers={"User-Agent": CFG["data"]["user_agent"]},
                 params={"action": "query", "titles": "Қазақстан", "prop": "revisions|pageprops", "rvlimit": 1,
                         "rvstart": CFG["data"]["wiki_cutoff_utc"], "rvdir": "older", "rvprop": "ids|timestamp",
                         "ppprop": "wikibase_item", "format": "json", "formatversion": 2}, timeout=30).json()
log.info(f"OK Wikipedia: {r['query']['pages'][0].get('revisions')} qid={r['query']['pages'][0].get('pageprops')}")

In [ ]:
# 4) GlotLID language ID
print(langid(["Қазақстан — Орталық Азиядағы мемлекет.", "Bu bir Türkçe cümledir.", "O'zbekiston Markaziy Osiyodagi davlat."]))

In [ ]:
# 5) Zero-shot scorer repos exist and share a tokenizer (required by Binoculars). No weights downloaded here.
from huggingface_hub import model_info
from transformers import AutoTokenizer
for pair, m in CFG["zeroshot"]["pairs"].items():
    try:
        for role in ("observer", "performer"):
            mi = model_info(m[role]); log.info(f"OK HF {pair}/{role}: {m[role]} pipeline={mi.pipeline_tag} license={(mi.card_data or {}).get('license')}")
        t1 = AutoTokenizer.from_pretrained(m["observer"]); t2 = AutoTokenizer.from_pretrained(m["performer"])
        same = t1.get_vocab() == t2.get_vocab()
        log.info(f"{'OK' if same else 'FAIL'} tokenizer identity for pair '{pair}': {same}")
    except Exception as e:
        log.error(f"FAIL HF pair {pair}: {e}  -> edit CFG['zeroshot']['pairs'] (FORCE_OVERWRITE=True, bump config_version)")

### Done
If every smoke test is OK, continue with **02_collect_human_wikipedia**. Paste any FAIL lines back to the research chat before proceeding.